Summary table for business question 4.3. Month is kept in the grain so the
same table also supports region and brand trends over time, not only the
single ranking the question asks for.

In [0]:
from pyspark.sql import functions as F

In [0]:
SOURCE_CATALOG_NAME = 'beverage_sales'
SOURCE_SCHEMA_NAME = 'gold'

TARGET_CATALOG_NAME = 'beverage_sales'
TARGET_SCHEMA_NAME = 'gold'
TARGET_TABLE_NAME = 'agg_sales_region_brand_month'

In [0]:
df_fact_sales = spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.fact_sales').alias('fs')
df_dim_region = spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.dim_region').alias('dr')
df_dim_brand = spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.dim_brand').alias('db')
df_dim_date = spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.dim_date').alias('dd')

In [0]:
df_agg = (
    df_fact_sales
    .join(
        F.broadcast(df_dim_region),
        on='region_key',
        how='inner'
    )
    .join(
        F.broadcast(df_dim_brand),
        on='brand_key',
        how='inner'
    )
    .join(
        F.broadcast(df_dim_date),
        on='date_key',
        how='inner'
    )
    .groupBy(
        'dr.region',
        'db.brand_name',
        'dd.year',
        'dd.month',
        'dd.year_month'
    )
    .agg(
        F.sum('fs.dollar_volume').alias('dollar_volume'),
        F.count('*').alias('record_count')
    )
)

In [0]:
df_agg\
    .write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')